# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Show the available record sets and their fields/columns

print("Available record sets (by @id and name):\n------------------")

# List all record sets and their associated fields and columns
for rs in dataset.metadata.record_sets:
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"  Field: @id={field.id}, name={field.name}, dataType={getattr(field, 'data_type', None)}")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"  Column: @id={col.id}, name={col.name}, dataType={getattr(col, 'data_type', None)}")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify available record_set @ids
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract each record set as a DataFrame
for rs_id in record_sets_ids:
    print(f"Extracting records for record_set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    if len(df) > 0:
        print(f"Columns for {rs_id}: {df.columns.tolist()}")
        display(df.head(2))

# Pick the main record set to focus analysis on. We'll choose the first one if multiple exist.
main_record_set_id = record_sets_ids[0]
print(f"Will use main_record_set_id = {main_record_set_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Determine a numeric field to analyze
# We'll automatically pick the first field or column of type number (Integer/Float) in the main record set

import numpy as np

main_record_set = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        main_record_set = rs
        break

numeric_field_id = None
numeric_columns = set()

for field in getattr(main_record_set, 'fields', []) + getattr(main_record_set, 'columns', []):
    dtype = getattr(field, 'data_type', None)
    if dtype is not None and ("Integer" in dtype or "Float" in dtype or "Number" in dtype):
        numeric_field_id = field.id
        numeric_columns.add(field.name)
        break

if not numeric_field_id:
    print("No numeric field found in main record set.")
else:
    print(f"Selecting numeric field for analysis: {numeric_field_id}")

df = dataframes[main_record_set_id].copy()

# Show column types
print("\nData sample and dtypes:")
display(df.head(3))
print(df.dtypes)

if numeric_field_id and numeric_columns:
    num_col = list(numeric_columns)[0]
    # Coerce to numeric
    df[num_col] = pd.to_numeric(df[num_col], errors='coerce')
    # Remove negative and extreme outliers if present
    threshold = df[num_col].mean() + 2 * df[num_col].std()
    filtered_df = df[df[num_col] < threshold]
    print(f"Filtered records where {num_col} < {threshold:.2f}:")
    display(filtered_df.head(3))

    # Normalize the numeric column
    filtered_df[f"{num_col}_normalized"] = (filtered_df[num_col] - filtered_df[num_col].mean()) / (filtered_df[num_col].std() + 1e-8)
    print(f"\nNormalized '{num_col}' (mean=0, std=1):")
    display(filtered_df[[num_col, f"{num_col}_normalized"]].head(3))

    # Try grouping by a categorical/text column if available
    group_field = None
    text_fields = []
    for field in getattr(main_record_set, 'fields', []) + getattr(main_record_set, 'columns', []):
        dtype = getattr(field, 'data_type', None)
        if dtype is not None and ("Text" in dtype or "String" in dtype):
            text_fields.append(field.name)
    if text_fields:
        group_field = text_fields[0]
        print(f"\nGrouping data by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[num_col].mean().to_frame(name=f"mean_{num_col}")
        display(grouped_df.head())
else:
    print("No valid numeric fields to analyze in main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_columns:
    num_col = list(numeric_columns)[0]
    if num_col in filtered_df:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[num_col].dropna(), kde=True)
        plt.title(f"Distribution of '{num_col}' in main record set")
        plt.xlabel(num_col)
        plt.show()

    # Boxplot by grouping variable if available
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        order = filtered_df[group_field].value_counts().index[:10]
        sns.boxplot(x=group_field, y=num_col, data=filtered_df, order=order)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Boxplot of '{num_col}' by '{group_field}'")
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> dataset, explored its record sets using the `@id` fields, and demonstrated how to extract, filter, and visualize tabular data using Croissant-compatible tools.

Key findings:
- The dataset records protein-related measurements from human extracellular vesicles.
- Through this pipeline, you can easily load and analyze data by referencing the Croissant schema, record sets, and fields by their persistent `@id`.

You are now set up to build further analysis or machine learning workflows using reproducibly referenced fields.